# Codify 201: store and search laws

Load sample laws into PostgreSQL, create embeddings for search, and retrieve
provisions through Python, HTTP and MCP.

Complete the setup in [Codify 101](codify-101.ipynb) and configure an embeddings
endpoint in `.env`. See [storage setup](../usage.md#store-search-compare) for the
settings. Start the local database and apply its migrations:

```bash
docker compose up -d --wait postgres
uv run alembic -c alembic.ini upgrade head
```

This notebook writes sample records to the database. Use a dedicated local database.
Embedding and search calls use your model endpoint and may incur charges.

In [1]:
import logging
from pathlib import Path

import langfuse  # noqa: F401, its import resets the logger quieted below
import structlog
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # the repo's .env; exported variables win
logging.getLogger("langfuse").setLevel(logging.ERROR)  # tracing is optional
structlog.configure(  # warnings only, uncoloured
    wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING),
    processors=[structlog.processors.add_log_level, structlog.dev.ConsoleRenderer(colors=False)],
)
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
FIXTURES = REPO / "tests" / "fixtures" / "synthetic"

## 1. Load the sample laws

Parse the four Atlantis (`xa`) acts and store them with `save_document`. Each law
has a work URI, each version has an expression URI, and provisions have their own rows.

Loading an existing expression URI returns the stored version. Bluebell uses the
run date for the expression, so running this on a later day creates new versions.
The CLI command `codify load bundle/ --embed` loads and embeds a saved bundle.

In [2]:
import json

from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

from codify.akn import parse_akn
from codify.akn.bluebell import bluebell_to_akn
from codify.settings import database_url
from codify.storage import save_document

engine = create_async_engine(database_url())
Session = async_sessionmaker(engine, expire_on_commit=False)

manifest = json.loads((FIXTURES / "xa" / "manifest.json").read_text())
versions = {}
async with Session() as session:
    for law in manifest["laws"]:
        akn_xml, _, errors = bluebell_to_akn(
            (FIXTURES / "xa" / law["file"]).read_text(),
            country="xa",
            doctype=law["doctype"],
            date=law["date"][:4],
            number=law["number"],
        )
        assert not errors, errors
        doc = parse_akn(akn_xml)
        versions[doc.frbr_work_uri] = await save_document(
            session,
            doc,
            jurisdiction_code="xa",
            law_title=law["short_title"],
            year=int(law["date"][:4]),
            number=law["number"],
            akn_xml=akn_xml,
        )
    await session.commit()
for uri, vid in versions.items():
    print(f"  {uri:<24} version {vid}")

  /akn/xa/act/1992/7       version 493e211a-cc5c-4ae8-9998-4cd6bf6c0e61
  /akn/xa/act/2001/3       version 2cb812ab-4c87-42cf-ba3d-2f32e3390d78
  /akn/xa/act/2011/5       version 828f2c84-b4c2-443d-ab94-55019de62ee1
  /akn/xa/act/2003/4       version 41fe2a99-106c-41fb-9d37-64ff882d7651


## 2. Create embeddings

Search uses an embedding for each provision. `embed_and_stamp` embeds a version
and records `embedded_at`. This example skips versions that already have that stamp.

The client uses an OpenAI-compatible embeddings endpoint. Set `EMBEDDING_*` in
`.env`; the URL and key can fall back to `LITELLM_BASE_URL` and `LITELLM_API_KEY`.

In [3]:
import os

from codify.embed.client import EmbeddingClient
from codify.storage import get_version
from codify.storage.embeddings import embed_and_stamp

embedder = EmbeddingClient(
    base_url=os.environ.get("EMBEDDING_BASE_URL") or os.environ["LITELLM_BASE_URL"],
    api_key=os.environ.get("EMBEDDING_API_KEY") or os.environ["LITELLM_API_KEY"],
    model=os.environ.get("EMBEDDING_MODEL", "gemini-embedding-2"),
)

async with Session() as session:
    for uri, vid in versions.items():
        if (await get_version(session, vid, with_akn=False)).embedded_at:
            print(f"  {uri:<24} already embedded")
            continue
        n = await embed_and_stamp(session, vid, client=embedder, path_context=False)
        print(f"  {uri:<24} {n} provisions embedded")
    await session.commit()

  /akn/xa/act/1992/7       58 provisions embedded


  /akn/xa/act/2001/3       7 provisions embedded


  /akn/xa/act/2011/5       5 provisions embedded


  /akn/xa/act/2003/4       7 provisions embedded


## 3. Read stored records

Use the storage helpers to list laws and retrieve a version.

In [4]:
from codify.storage.laws import list_laws

async with Session() as session:
    rows, _ = await list_laws(session, jurisdictions=["xa"])
    for law in rows:
        print(f"  {law.year}  {law.title:<36} {law.frbr_work_uri}")
    version = await get_version(session, versions["/akn/xa/act/1992/7"], with_akn=False)
    print("\nversion", version.id, "|", version.language, "|", version.expression_date)

  2011  Legislation (Amendment) Act, 2011    /akn/xa/act/2011/5
  2003  Penalties Act, 2003                  /akn/xa/act/2003/4
  2001  Freedom of Information Act, 2001     /akn/xa/act/2001/3
  1992  Legislation Act, 1992                /akn/xa/act/1992/7

version 493e211a-cc5c-4ae8-9998-4cd6bf6c0e61 | eng | 2026-09-18


## 4. Search provisions

`retrieve` combines keyword search (BM25) with embedding similarity, ranking the
results using reciprocal rank fusion. You can search a jurisdiction, law or version.
A jurisdiction search uses the latest version of each law.

In [5]:
from codify.retrieve.hybrid import retrieve


async def search(query, **scope):
    async with Session() as session:
        matches = await retrieve(session, query, embedding_client=embedder, k=5, **scope)
    print(f"{query!r}")
    for m in matches:
        print(f"  {m.rrf_score:.4f}  {m.akn_eid:<26} {m.text[:80]}")


await search("right of access to the official text of the law", jurisdiction_code="xa")

'right of access to the official text of the law'
  0.0328  sec_12__subsec_1__content  Every person has the right of access to the official text of any law in force, i
  0.0310  sec_12__subsec_2__content  No fee shall be charged for access to the official text in electronic form, but 
  0.0308  sec_2__subsec_1__point_e   "official text", in relation to a law, means the text of the law as enacted or m
  0.0301  sec_5__subsec_1__content   The Publishing Authority shall maintain the official text of every law in a stat
  0.0292  sec_3__content             Nothing in this Act requires a public authority to communicate information that 


In [6]:
await search("maximum fine for an offence", jurisdiction_code="xa")
await search("maximum fine for an offence", version_id=versions["/akn/xa/act/1992/7"])

'maximum fine for an offence'
  0.0328  sec_4__content             The following penalties apply to the offences specified: OffenceMaximum fineMaxi
  0.0320  sec_15__subsec_2__content  A person who is guilty of an offence under subsection (1) is liable on convictio
  0.0318  sec_3__subsec_1__content   Where an offence is specified in the Schedule, the court shall, upon conviction,
  0.0315  sec_14__content            A person who wilfully obstructs the Publishing Authority, or an officer of the O
  0.0293  sec_16__content            Where an offence under this Act committed by a body corporate is proved to have 


'maximum fine for an offence'
  0.0328  sec_15__subsec_2__content  A person who is guilty of an offence under subsection (1) is liable on convictio
  0.0323  sec_14__content            A person who wilfully obstructs the Publishing Authority, or an officer of the O
  0.0306  sec_15__subsec_1__content  A person who, in the Register or in the official text, makes or causes to be mad
  0.0304  sec_16__content            Where an offence under this Act committed by a body corporate is proved to have 
  0.0303  sec_12__subsec_2__content  No fee shall be charged for access to the official text in electronic form, but 


These queries illustrate why both methods are useful: keyword search matches
specific terms, while embeddings can find related wording.

In [7]:
await search("can I look at the register without paying", jurisdiction_code="xa")
await search(
    "when does an Act come into force if the Minister appoints no day", jurisdiction_code="xa"
)

'can I look at the register without paying'
  0.0307  sec_8__subsec_2__content   The Register shall be open to inspection by any person, free of charge, at all r
  0.0306  sec_15__subsec_1__content  A person who, in the Register or in the official text, makes or causes to be mad
  0.0291  sec_2__subsec_1__point_h   "the Register" means the register of legislation kept under section 8;
  0.0289  sec_10__subsec_1__content  The functions of the Publishing Authority are to publish the laws in accordance 
  0.0281  sec_4__subsec_2__content   The reproduction of a printed page as an image shall not, of itself, satisfy the


'when does an Act come into force if the Minister appoints no day'
  0.0325  sec_1__subsec_2__content   This Act shall come into operation on such day as the Minister may, by notice pu
  0.0306  sec_1__subsec_3__content   Where a provision of this Act has not been brought into operation within two yea
  0.0295  sec_2__subsec_1__point_b   "commencement", in relation to a provision, means the day on which the provision
  0.0293  sec_8__subsec_1__point_b   the date of its enactment or making and the date of its commencement; and
  0.0290  sec_3__subsec_1__content   This Act applies to every law in force on the commencement of this section and t


## 5. Use the HTTP API

`codify serve` exposes the library through FastAPI. This example calls the app
within the notebook, without starting a separate server. The API schema is in
`contract/openapi.json`.

In [8]:
from httpx import ASGITransport, AsyncClient

from codify.serve import create_app

app = create_app()
async with AsyncClient(transport=ASGITransport(app=app), base_url="http://codify") as http:
    async with app.router.lifespan_context(app):
        r = await http.get("/jurisdictions/xa")
        print(r.status_code, r.json()["name"], r.json()["languages"])
        r = await http.get("/laws", params={"jurisdiction": "xa"})
        print([law["title"] for law in r.json()["items"]])
        r = await http.get(
            "/search", params={"q": "register of legislation", "jurisdiction": "xa", "k": 3}
        )
        for m in r.json()["matches"]:
            print(f"  {m['score']:.4f}  {m['eid']:<26} {m['text'][:70]}")

200 Atlantis ['eng']
['Legislation (Amendment) Act, 2011', 'Penalties Act, 2003', 'Freedom of Information Act, 2001', 'Legislation Act, 1992']


  0.0328  sec_2__subsec_1__point_h   "the Register" means the register of legislation kept under section 8;
  0.0320  sec_17__subsec_2__point_a  prescribe the form and manner of publication of laws and of the Regist
  0.0310  sec_8__subsec_2__content   The Register shall be open to inspection by any person, free of charge


## 6. Use MCP tools

`codify mcp` lets an MCP client search, retrieve documents and run comparisons.
This example connects within the notebook. For a separate client, start the server
with `uv run --extra mcp codify mcp` over stdio.

See the [interfaces guide](../interfaces.md) for server setup.

In [9]:
from mcp import Client

from codify.mcp import create_server

async with Client(create_server()) as mcp:
    print([t.name for t in (await mcp.list_tools()).tools])
    result = await mcp.call_tool(
        "search_provisions",
        {"query": "appeal against a decision of the Authority", "jurisdiction": "xa", "k": 3},
    )
    for m in result.structured_content["matches"]:
        print(f"  {m['score']:.4f}  {m['eid']:<26} {m['text'][:70]}")

['search_provisions', 'list_laws', 'get_law', 'get_version', 'list_jurisdictions', 'get_jurisdiction', 'compare_versions']


  0.0323  sec_5__content             A person aggrieved by a decision under this Act may appeal to the Trib
  0.0323  sec_13__subsec_1__content  A person aggrieved by a decision of the Publishing Authority under thi
  0.0318  sec_13__subsec_2__content  On an appeal under this section the Tribunal may confirm, vary or set 


In [10]:
await engine.dispose()

Continue with [jurisdiction configuration](codify-301a.ipynb) or
[document comparison](codify-301.ipynb).